# SD4ft-Miner: Hotel cancellations

We compare cancellation rates between City Hotel and Resort Hotel.

**Question:** Which booking subgroups show unusually large differences?

**Hypothesis:** Some subgroups have a City/Resort rate ratio of at least 1.8.
The overall ratio is about 1.5.

In [12]:
from pathlib import Path

import pandas as pd
from cleverminer import cleverminer


path = Path("../data/processed/hotel_bookings_categorized.csv")

columns = [
    "hotel", 
    "is_canceled", 
    "lead_time_cat",
    "market_segment", 
    "customer_type", 
    "season",
]
df = pd.read_csv(path, dtype=str)[columns]

rates = df.groupby("hotel")["is_canceled"].apply(
    lambda values: values.eq("1").mean()
)
city_rate = rates["City Hotel"]
resort_rate = rates["Resort Hotel"]

print(f"Bookings: {len(df):,}")
print(f"Overall: City {city_rate:.1%}, Resort {resort_rate:.1%}")
print(f"Overall City / Resort ratio: {city_rate / resort_rate:.2f}")

Bookings: 118,558
Overall: City 41.9%, Resort 28.0%
Overall City / Resort ratio: 1.50


## SD4ft-Miner

We compare the same subgroup in both hotels.
Each subgroup has two characteristics.
We require at least 200 cancellations per hotel and a ratio of 1.8.

In [ ]:
# Each subgroup contains exactly two characteristics.
def one_value(name, value):
    return {
        "attributes": [
            {"name": name, "type": "one", "value": value}
        ],
        "minlen": 1, 
        "maxlen": 1, 
        "type": "con",
    }


miner = cleverminer(
    df=df,
    proc="SD4ftMiner",
    quantifiers={
        "Base1": 200,
        "Base2": 200,
        "Ratioconf": 1.8
    },
    ante={
        "attributes": [
            {"name": "lead_time_cat", "type": "seq", "minlen": 1, "maxlen": 2},
            {"name": "market_segment", "type": "subset", "minlen": 1, "maxlen": 1},
            {"name": "customer_type", "type": "subset", "minlen": 1, "maxlen": 1},
            {"name": "season", "type": "subset", "minlen": 1, "maxlen": 1},
        ],
        "minlen": 2, 
        "maxlen": 2, 
        "type": "con",
    },
    succ=one_value("is_canceled", "1"),
    frst=one_value("hotel", "City Hotel"),
    scnd=one_value("hotel", "Resort Hotel"),
    opts={
        "no_automatic_data_conversions": True,
        "disable_progressbar": True
    },
)

Cleverminer version 1.2.8.
Progressbar will not be shown.
No automatic data conversions will be made.
Progressbar will not be shown.
No automatic data conversions will be made.
Starting data preparation ...
Encoding columns into bit-form...
Encoding columns into bit-form...done
Data preparation finished.
Progressbar will not be shown.
No automatic data conversions will be made.
Will go for  SD4ftMiner
Starting to mine rules.
Done. Total verifications : 159, rules 18, times: prep 0.09sec, processing 0.04sec


## Results

We show two rules from different market segments.
They are selected by the largest difference in cancellation rates.

In [14]:
miner.print_summary()

# Compare rules in different market segments (no subgroup overlap).
ranked = []
for rule_id in range(1, miner.get_rulecount() + 1):
    text = miner.get_ruletext(rule_id).split("=>")[0]
    if "market_segment(" not in text:
        continue

    segment = text.split("market_segment(", 1)[1].split(")", 1)[0]
    city = miner.get_fourfold(rule_id, order=1)
    resort = miner.get_fourfold(rule_id, order=2)
    city_rate = city[0] / (city[0] + city[1])
    resort_rate = resort[0] / (resort[0] + resort[1])
    ranked.append((city_rate - resort_rate, rule_id, segment))

shown = set()
for gap, rule_id, segment in sorted(ranked, reverse=True):
    if segment in shown:
        continue

    shown.add(segment)
    city = miner.get_fourfold(rule_id, order=1)
    resort = miner.get_fourfold(rule_id, order=2)
    p_city = city[0] / (city[0] + city[1])
    p_resort = resort[0] / (resort[0] + resort[1])

    print(f"\nRule {rule_id}: {segment}")
    print(f"City: {p_city:.1%} | Resort: {p_resort:.1%}")
    print(f"Ratio: {p_city / p_resort:.2f} | Gap: {gap * 100:.1f} pp")
    miner.print_rule(rule_id)

    if len(shown) == 2:
        break

print("\nInterpretation: descriptive associations, not causal effects.")


CleverMiner task processing summary:

Task type : SD4ftMiner
Number of verifications : 159
Number of rules : 18
Total time needed : 00h 00m 00s
Time of data preparation : 00h 00m 00s
Time of rule mining : 00h 00m 00s


Rule 9: Groups
City: 81.5% | Resort: 42.0%
Ratio: 1.94 | Gap: 39.5 pp


Rule id : 9

Base1 :  3867 Base2 :   928  Relative base 1 : 0.033 Relative base 2 : 0.008 CONF1 : 0.815  CONF2 : +0.420  Delta Conf : +0.395 Ratio Conf : +1.942

Cedents:
  antecedent : lead_time_cat(5_181-365) & market_segment(Groups)
  succcedent : is_canceled(1)
  condition  : ---
  first set  : hotel(City Hotel)
  second set : hotel(Resort Hotel)

Fourfold tables:
FRST|  S  |  ¬S |  SCND|  S  |  ¬S |
----|-----|-----|  ----|-----|-----| 
 A  | 3867|  877|   A  |  928| 1283|
----|-----|-----|  ----|-----|-----|
¬A  |29194|44956|  ¬A  |10182|27271|
----|-----|-----|  ----|-----|-----|


Rule 14: Offline TA/TO
City: 55.9% | Resort: 17.4%
Ratio: 3.21 | Gap: 38.4 pp


Rule id : 14

Base1 :  5169 Base

## Interpretation

- **Groups, 181–365 days ahead:** 81.5% canceled in City Hotel vs 42.0% in Resort Hotel (ratio 1.94).
- **Offline TA/TO, Transient:** 55.9% canceled in City Hotel vs 17.4% in Resort Hotel (ratio 3.21).

Both ratios exceed the overall ratio of 1.50.
These are associations, not causal effects.